# Version 1 - BILSTM

In [1]:
import numpy as np
import pandas as pd
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input
from tensorflow.keras.models import Model

# ============================================

# 1. LOAD DATASET

# ============================================

df = pd.read_csv('/kaggle/input/datasets/nlztrk/eduqg-dataset-llm-science-exam-format-34k/eduqg_llm_formatted.csv')
questions = df['prompt'].dropna().tolist()

# ============================================

# 2. CLEAN QUESTIONS

# ============================================

def clean_question(q):
    q = q.lower()
    q = q.replace("which of the following", "")
    q = q.replace("select the correct answer", "")
    q = q.replace("choose the correct option", "")
    return q.strip()

questions = [clean_question(q) for q in questions if len(q.split()) > 4]

# ============================================

# 3. BLOOM LABELING

# ============================================

def assign_bloom_level(q):
    q = q.lower()
    
    if "design" in q or "develop" in q:
        return "Create"
    if "justify" in q or "why" in q:
        return "Evaluate"
    if "compare" in q or "difference" in q:
        return "Analyze"
    if "calculate" in q or "solve" in q:
        return "Apply"
    if "explain" in q or "describe" in q:
        return "Understand"
    if "define" in q or "what is" in q:
        return "Remember"
    
    return "Understand"


labels = [assign_bloom_level(q) for q in questions]

# ============================================

# 4. PREPROCESSING

# ============================================

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = re.sub(r'[^a-zA-Z ]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

questions_clean = [preprocess(q) for q in questions]

# ============================================

# 5. BALANCE DATASET (FIXED)

# ============================================

df_bal = pd.DataFrame({'q': questions_clean, 'label': labels})

MAX_SAMPLES_PER_CLASS = 200

df_bal = df_bal.groupby('label').apply(
lambda x: x.sample(min(len(x), MAX_SAMPLES_PER_CLASS), random_state=42)
).reset_index(drop=True)

questions_clean = df_bal['q'].tolist()
labels = df_bal['label'].tolist()

# ============================================

# 6. TOKENIZATION

# ============================================

MAX_LEN = 50
VOCAB_SIZE = 20000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(questions_clean)

sequences = tokenizer.texts_to_sequences(questions_clean)
padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

# ============================================

# 7. LABEL ENCODING

# ============================================

le = LabelEncoder()
y = le.fit_transform(labels)

NUM_CLASSES = len(le.classes_)
IDX_TO_LABEL = {i: label for i, label in enumerate(le.classes_)}

# ============================================

# 8. TRAIN TEST SPLIT

# ============================================

X_train, X_test, y_train, y_test = train_test_split(
padded, y, test_size=0.2, random_state=42
)

y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

# ============================================

# 9. CLASS WEIGHTS

# ============================================

class_weights = compute_class_weight(
class_weight='balanced',
classes=np.unique(y_train),
y=y_train
)

class_weights = dict(enumerate(class_weights))

# ============================================

# 10. MODEL

# ============================================

input_layer = Input(shape=(MAX_LEN,))
x = Embedding(VOCAB_SIZE, 100)(input_layer)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Conv1D(64, 3, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(
loss='categorical_crossentropy',
optimizer='adam',
metrics=['accuracy']
)

# ============================================

# 11. TRAIN

# ============================================

model.fit(
X_train, y_train_cat,
validation_data=(X_test, y_test_cat),
epochs=10,
batch_size=32,
class_weight=class_weights
)

# ============================================

# 12. PREDICTION FUNCTION

# ============================================

def predict_bloom(question):
    q = clean_question(question)
    q = preprocess(q)
    seq = tokenizer.texts_to_sequences([q])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    pred = model.predict(pad, verbose=0)
    return IDX_TO_LABEL[np.argmax(pred)]

# ============================================

# 13. PROCESS TXT FILE

# ============================================

def process_txt(file_path):
    results = []
    
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        match = re.match(r"(Q\d+)\.\s*(.*)", line)
        
        if match:
            qno = match.group(1)
            question = match.group(2)
            
            btl = predict_bloom(question)
            results.append([qno, question, btl])
    
    return results


# ============================================

# 14. DISPLAY OUTPUT

# ============================================

def display_results(results):
    df_out = pd.DataFrame(results, columns=["Qno", "Question", "BTL Level"])
    print("\n===== BTL CLASSIFICATION RESULTS =====\n")
    print(df_out.to_string(index=False))

# ============================================

# 15. RUN

# ============================================

file_path = '/kaggle/input/datasets/lakshmir03/btl-que/btl_questions.txt'

results = process_txt(file_path)
display_results(results)


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2026-04-04 12:19:03.184086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775305143.436686      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775305143.510269      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775305144.146695      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775305144.146734      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.1519 - loss: 1.7598 - val_accuracy: 0.1552 - val_loss: 1.7793
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.1921 - loss: 1.7708 - val_accuracy: 0.1638 - val_loss: 1.7938
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.2139 - loss: 1.7224 - val_accuracy: 0.2241 - val_loss: 1.7922
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.3557 - loss: 1.6689 - val_accuracy: 0.2586 - val_loss: 1.7417
Epoch 5/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.4013 - loss: 1.5146 - val_accuracy: 0.4483 - val_loss: 1.4278
Epoch 6/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.5429 - loss: 1.1669 - val_accuracy: 0.4741 - val_loss: 1.3874
Epoch 7/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.7710 - loss: 0.6878 - val_accuracy: 0.4828 - val_loss: 1.3900
Epoch 8/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.7626 - loss: 0.4437 - val_accuracy: 0.4914 - v

# Version 2
✔ Better labeling
✔ Balanced dataset (correctly)
✔ Minimal preprocessing (no information loss)
✔ Confidence scores
✔ Proper metrics
✔ Robust TXT parsing

In [2]:
# ============================================

# FINAL BTL CLASSIFIER (IMPROVED VERSION)

# ============================================

import numpy as np
import pandas as pd
import re
import nltk
nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input
from tensorflow.keras.models import Model

# ============================================

# 1. LOAD DATASET

# ============================================

df = pd.read_csv('/kaggle/input/datasets/nlztrk/eduqg-dataset-llm-science-exam-format-34k/eduqg_llm_formatted.csv')
questions = df['prompt'].dropna().tolist()

# ============================================

# 2. MINIMAL CLEANING (IMPORTANT FIX)

# ============================================

questions = [q.lower().strip() for q in questions if len(q.split()) > 4]

# ============================================

# 3. IMPROVED BLOOM LABELING

# ============================================

def assign_bloom_level(q):
    q = q.lower()
    
    if any(x in q for x in ["design", "develop", "formulate", "construct", "create"]):
        return "Create"
    
    if any(x in q for x in ["justify", "evaluate", "critique", "assess", "what criteria", "why", "opinion"]):
        return "Evaluate"
    
    if any(x in q for x in ["compare", "difference", "analyze", "analyse", "distinguish", "relationship"]):
        return "Analyze"
    
    if any(x in q for x in ["calculate", "solve", "use", "demonstrate", "implement"]):
        return "Apply"
    
    if any(x in q for x in ["define", "what is", "how many", "list", "identify", "name"]):
        return "Remember"
    
    if any(x in q for x in ["explain", "describe", "interpret", "which", "what would happen"]):
        return "Understand"
    
    return "Understand"


labels = [assign_bloom_level(q) for q in questions]

# ============================================

# 4. OPTIONAL DATA AUGMENTATION (SMALL BOOST)

# ============================================

extra_questions = [
"Explain how neural networks work",
"Compare supervised and unsupervised learning",
"Design a system for traffic prediction",
"Evaluate the impact of climate change"
]

questions.extend(extra_questions)
labels.extend([assign_bloom_level(q) for q in extra_questions])

# ============================================

# 5. TOKENIZATION

# ============================================

MAX_LEN = 50
VOCAB_SIZE = 20000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(questions)

sequences = tokenizer.texts_to_sequences(questions)
padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

# ============================================

# 6. LABEL ENCODING

# ============================================

le = LabelEncoder()
y = le.fit_transform(labels)

NUM_CLASSES = len(le.classes_)
IDX_TO_LABEL = {i: label for i, label in enumerate(le.classes_)}

# ============================================

# 7. TRAIN TEST SPLIT

# ============================================

X_train, X_test, y_train, y_test = train_test_split(
padded, y, test_size=0.2, random_state=42
)

y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

# ============================================

# 8. CLASS WEIGHTS

# ============================================

class_weights = compute_class_weight(
class_weight='balanced',
classes=np.unique(y_train),
y=y_train
)

class_weights = dict(enumerate(class_weights))

# ============================================

# 9. MODEL (BiLSTM + CNN)

# ============================================

input_layer = Input(shape=(MAX_LEN,))
x = Embedding(VOCAB_SIZE, 100)(input_layer)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Conv1D(64, 3, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(
loss='categorical_crossentropy',
optimizer='adam',
metrics=['accuracy']
)

model.summary()

# ============================================

# 10. TRAIN

# ============================================

model.fit(
X_train, y_train_cat,
validation_data=(X_test, y_test_cat),
epochs=10,
batch_size=32,
class_weight=class_weights
)

# ============================================

# 11. PREDICTION FUNCTION (WITH CONFIDENCE)

# ============================================

def predict_bloom(question):
    q = question.lower()
    seq = tokenizer.texts_to_sequences([q])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    
    pred = model.predict(pad, verbose=0)[0]
    
    label = IDX_TO_LABEL[np.argmax(pred)]
    confidence = np.max(pred)
    
    return label, round(float(confidence), 2)


# ============================================

# 12. PROCESS TXT FILE

# ============================================

def process_txt(file_path):
    results = []
    
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Supports both Q1. and 1. formats
        match = re.match(r"(Q?\d+)\.\s*(.*)", line)
        
        if match:
            qno = match.group(1)
            question = match.group(2)
            
            label, confidence = predict_bloom(question)
            results.append([qno, question, label, confidence])
    
    return results


# ============================================

# 13. DISPLAY OUTPUT

# ============================================

def display_results(results):
    df_out = pd.DataFrame(results, columns=["Qno", "Question", "BTL Level", "Confidence"])
    
    print("\n===== BTL CLASSIFICATION RESULTS =====\n")
    print(df_out.to_string(index=False))

# ============================================

# 14. RUN

# ============================================

file_path = '/kaggle/input/datasets/lakshmir03/btl-que/btl_questions.txt'

results = process_txt(file_path)
display_results(results)


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2026-04-04 12:25:03.561359: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775305503.756256      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775305503.815174      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775305504.249083      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775305504.249131      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 50, 100)        │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 256)        │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 48, 64)         │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,288,262 (8.73 MB)

 Trainable params: 2,288,262 (8.73 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


I0000 00:00:1775305532.269042     145 cuda_dnn.cc:529] Loaded cuDNN version 91002


83/83 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.1641 - loss: 1.8577 - val_accuracy: 0.6471 - val_loss: 1.4231
Epoch 2/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5067 - loss: 1.6005 - val_accuracy: 0.5551 - val_loss: 1.3382
Epoch 3/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5023 - loss: 1.0650 - val_accuracy: 0.6139 - val_loss: 1.2111
Epoch 4/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5942 - loss: 0.5509 - val_accuracy: 0.6772 - val_loss: 0.9552
Epoch 5/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7930 - loss: 0.3227 - val_accuracy: 0.7647 - val_loss: 0.7656
Epoch 6/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9314 - loss: 0.1390 - val_accuracy: 0.9050 - val_loss: 0.4210
Epoch 7/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9825 - loss: 0.0464 - val_accuracy: 0.8718 - val_loss: 0.4977
Epoch 8/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9866 - loss: 0.0345 - val_accuracy: 0.8778 - val_loss: 0.

# Ver 3


In [5]:
# ============================================

# FINAL BTL CLASSIFIER (IMPROVED VERSION)

# ============================================

import numpy as np
import pandas as pd
import re
import nltk
nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input
from tensorflow.keras.models import Model

# ============================================

# 1. LOAD DATASET

# ============================================

df = pd.read_csv('/kaggle/input/datasets/nlztrk/eduqg-dataset-llm-science-exam-format-34k/eduqg_llm_formatted.csv')
questions = df['prompt'].dropna().tolist()

# ============================================

# 2. MINIMAL CLEANING (IMPORTANT FIX)

# ============================================

questions = [q.lower().strip() for q in questions if len(q.split()) > 4]

# ============================================

# 3. IMPROVED BLOOM LABELING

# ============================================

def assign_bloom_level(q):
    q = q.lower()
    
    if any(x in q for x in ["design", "develop", "formulate", "construct", "create"]):
        return "Create"
    
    if any(x in q for x in ["justify", "evaluate", "critique", "assess", "what criteria", "why", "opinion"]):
        return "Evaluate"
    
    if any(x in q for x in ["compare", "difference", "analyze", "analyse", "distinguish", "relationship"]):
        return "Analyze"
    
    if any(x in q for x in ["calculate", "solve", "use", "demonstrate", "implement"]):
        return "Apply"
    
    if any(x in q for x in ["define", "what is", "how many", "list", "identify", "name"]):
        return "Remember"
    
    if any(x in q for x in ["explain", "describe", "interpret", "which", "what would happen"]):
        return "Understand"
    
    return "Understand"
    

labels = [assign_bloom_level(q) for q in questions]

# ============================================

# 4. OPTIONAL DATA AUGMENTATION (SMALL BOOST)

# ============================================

extra_questions = [
"Explain how neural networks work",
"Compare supervised and unsupervised learning",
"Design a system for traffic prediction",
"Evaluate the impact of climate change"
]

questions.extend(extra_questions)
labels.extend([assign_bloom_level(q) for q in extra_questions])

# ============================================

# 5. TOKENIZATION

# ============================================

MAX_LEN = 50
VOCAB_SIZE = 20000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(questions)

sequences = tokenizer.texts_to_sequences(questions)
padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

# ============================================

# 6. LABEL ENCODING

# ============================================

le = LabelEncoder()
y = le.fit_transform(labels)

NUM_CLASSES = len(le.classes_)
IDX_TO_LABEL = {i: label for i, label in enumerate(le.classes_)}

# ============================================

# 7. TRAIN TEST SPLIT

# ============================================

X_train, X_test, y_train, y_test = train_test_split(
padded, y, test_size=0.2, random_state=42
)

y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

# ============================================

# 8. CLASS WEIGHTS

# ============================================

class_weights = compute_class_weight(
class_weight='balanced',
classes=np.unique(y_train),
y=y_train
)

class_weights = dict(enumerate(class_weights))

# ============================================

# 9. MODEL (BiLSTM + CNN)

# ============================================

input_layer = Input(shape=(MAX_LEN,))
x = Embedding(VOCAB_SIZE, 100)(input_layer)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Conv1D(64, 3, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(
loss='categorical_crossentropy',
optimizer='adam',
metrics=['accuracy']
)

model.summary()

# ============================================

# 10. TRAIN

# ============================================

model.fit(
X_train, y_train_cat,
validation_data=(X_test, y_test_cat),
epochs=30,
batch_size=32,
class_weight=class_weights
)

# ============================================

# 11. PREDICTION FUNCTION (WITH CONFIDENCE)

# ============================================

def predict_bloom(question):
    q = question.lower()
    seq = tokenizer.texts_to_sequences([q])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    
    pred = model.predict(pad, verbose=0)[0]
    
    label = IDX_TO_LABEL[np.argmax(pred)]
    confidence = np.max(pred)
    
    return label, round(float(confidence), 2)

# ============================================

# 12. PROCESS TXT FILE

# ============================================

def process_txt(file_path):
    results = []
    
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Supports both Q1. and 1. formats
        match = re.match(r"(Q?\d+)\.\s*(.*)", line)
        
        if match:
            qno = match.group(1)
            question = match.group(2)
            
            label, confidence = predict_bloom(question)
            results.append([qno, question, label, confidence])
    
    return results


# ============================================

# 13. DISPLAY OUTPUT

# ============================================

def display_results(results):
    df_out = pd.DataFrame(results, columns=["Qno", "Question", "BTL Level", "Confidence"])
    
    print("\n===== BTL CLASSIFICATION RESULTS =====\n")
    print(df_out.to_string(index=False))
    

# ============================================

# 14. RUN

# ============================================

file_path = '/kaggle/input/datasets/lakshmir03/btl-que/btl_questions.txt'

results = process_txt(file_path)
display_results(results)


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 50, 100)        │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 50, 256)        │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 48, 64)         │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,288,262 (8.73 MB)

 Trainable params: 2,288,262 (8.73 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.1840 - loss: 1.8543 - val_accuracy: 0.7149 - val_loss: 1.5969
Epoch 2/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5273 - loss: 1.6210 - val_accuracy: 0.4299 - val_loss: 1.4323
Epoch 3/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5878 - loss: 1.0753 - val_accuracy: 0.2142 - val_loss: 1.3520
Epoch 4/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6413 - loss: 0.6552 - val_accuracy: 0.7421 - val_loss: 0.8155
Epoch 5/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8828 - loss: 0.3244 - val_accuracy: 0.7345 - val_loss: 0.7778
Epoch 6/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9426 - loss: 0.2619 - val_accuracy: 0.8537 - val_loss: 0.6420
Epoch 7/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9874 - loss: 0.0754 - val_accuracy: 0.8356 - val_loss: 0.6975
Epoch 8/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9881 - loss: 0.0379 - val_accuracy: 0.7949 - v

# BERT

In [11]:
# ============================================
# BERT BTL CLASSIFIER (TXT INPUT - FIXED GPU)
# ============================================

!pip install transformers -q

import numpy as np
import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# ============================================
# DEVICE SETUP (FIX)
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_csv('/kaggle/input/datasets/nlztrk/eduqg-dataset-llm-science-exam-format-34k/eduqg_llm_formatted.csv')
questions = df['prompt'].dropna().tolist()
questions = [q.strip() for q in questions if len(q.split()) > 4]

# ============================================
# LABELING
# ============================================

def assign_bloom_level(q):
    q = q.lower()

    if any(x in q for x in ["design", "develop", "construct", "create"]):
        return "Create"
    if any(x in q for x in ["assess", "justify", "evaluate", "criteria", "why", "solution"]):
        return "Evaluate"
    if any(x in q for x in ["compare", "difference", "analyze"]):
        return "Analyze"
    if any(x in q for x in ["calculate", "solve", "use"]):
        return "Apply"
    if any(x in q for x in ["define", "what is", "how many"]):
        return "Remember"
    return "Understand"

labels = [assign_bloom_level(q) for q in questions]

# ============================================
# ENCODE LABELS
# ============================================

le = LabelEncoder()
y = le.fit_transform(labels)
IDX_TO_LABEL = {i: label for i, label in enumerate(le.classes_)}

# ============================================
# SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    questions, y, test_size=0.2, random_state=42
)

# ============================================
# TOKENIZER
# ============================================

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=64)

train_encodings = tokenize(X_train)

# ============================================
# DATASET CLASS
# ============================================

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, y_train)

# ============================================
# MODEL
# ============================================

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(le.classes_)
)

model.to(device)

# ============================================
# TRAINING
# ============================================

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

# ============================================
# PREDICTION (FIXED DEVICE)
# ============================================

def predict_bloom(question):
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True)

    inputs = {k: v.to(device) for k, v in inputs.items()}  # FIX

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    label = IDX_TO_LABEL[np.argmax(probs)]
    confidence = round(float(np.max(probs)), 2)

    return label, confidence

# ============================================
# TXT PROCESSING
# ============================================

def process_txt(file_path):
    results = []

    with open(file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        match = re.match(r"(Q?\d+)\.\s*(.*)", line)

        if match:
            qno = match.group(1)
            question = match.group(2)

            label, confidence = predict_bloom(question)
            results.append([qno, question, label, confidence])

    return results

# ============================================
# DISPLAY
# ============================================

def display_results(results):
    df_out = pd.DataFrame(results, columns=["Qno", "Question", "BTL Level", "Confidence"])
    print("\n===== BTL CLASSIFICATION RESULTS (BERT) =====\n")
    print(df_out.to_string(index=False))

# ============================================
# RUN
# ============================================

file_path = '/kaggle/input/datasets/lakshmir03/btl-que/btl_questions.txt'

results = process_txt(file_path)
display_results(results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Step,Training Loss
10,2.780947
20,1.892143
30,1.713402
40,1.973752
50,2.049969
60,1.772464
70,1.576258
80,1.405034
90,1.133842
100,1.288535


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== BTL CLASSIFICATION RESULTS (BERT) =====

Qno                                                                             Question  BTL Level  Confidence
 Q1 Which of the following groups or bodies did not offer direct relief to needy people? Understand        0.97
 Q2                          What criteria would you use to assess today's stock market?      Apply        0.89
 Q3                                          Can you see a possible solution to famines? Understand        0.97
 Q4                                                  What would result in deforestation? Understand        0.93
 Q5                                                            How many eggs in a dozen?   Remember        0.67


In [12]:
# ============================================
# FINAL HYBRID BTL CLASSIFIER (BERT + RULES)
# ============================================

!pip install transformers -q

import numpy as np
import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# ============================================
# DEVICE
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================
# LOAD DATA
# ============================================

df = pd.read_csv('/kaggle/input/datasets/nlztrk/eduqg-dataset-llm-science-exam-format-34k/eduqg_llm_formatted.csv')
questions = df['prompt'].dropna().tolist()
questions = [q.strip() for q in questions if len(q.split()) > 4]

# ============================================
# STRONG LABELING (TRAINING)
# ============================================

def assign_bloom_level(q):
    q = q.lower()

    if any(x in q for x in ["design", "develop", "construct", "create", "propose", "solution"]):
        return "Create"
    if any(x in q for x in ["assess", "justify", "evaluate", "criteria", "why"]):
        return "Evaluate"
    if any(x in q for x in ["compare", "difference", "analyze", "distinguish"]):
        return "Analyze"
    if any(x in q for x in ["calculate", "solve", "use", "implement"]):
        return "Apply"
    if any(x in q for x in ["define", "what is", "how many", "identify"]):
        return "Remember"
    return "Understand"

labels = [assign_bloom_level(q) for q in questions]

# ============================================
# ENCODE
# ============================================

le = LabelEncoder()
y = le.fit_transform(labels)
IDX_TO_LABEL = {i: label for i, label in enumerate(le.classes_)}

# ============================================
# SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    questions, y, test_size=0.2, random_state=42
)

# ============================================
# TOKENIZER
# ============================================

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=64)

train_encodings = tokenize(X_train)

# ============================================
# DATASET
# ============================================

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, y_train)

# ============================================
# MODEL
# ============================================

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(le.classes_)
)

model.to(device)

# ============================================
# TRAIN
# ============================================

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

# ============================================
# BERT PREDICTION
# ============================================

def bert_predict(question):
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    label = IDX_TO_LABEL[np.argmax(probs)]
    confidence = float(np.max(probs))

    return label, confidence

# ============================================
# HYBRID SYSTEM (FINAL LOGIC)
# ============================================

def hybrid_predict(question):
    q = question.lower()

    # ===== HARD RULES (HIGH CONFIDENCE) =====
    if any(x in q for x in ["design", "propose", "create", "solution"]):
        return "Create", 0.99

    if any(x in q for x in ["assess", "criteria", "justify", "evaluate"]):
        return "Evaluate", 0.99

    if any(x in q for x in ["compare", "difference", "distinguish"]):
        return "Analyze", 0.95

    if any(x in q for x in ["calculate", "solve"]):
        return "Apply", 0.95

    if any(x in q for x in ["define", "how many", "what is"]):
        return "Remember", 0.95

    # ===== FALLBACK TO BERT =====
    label, conf = bert_predict(question)

    # confidence calibration
    conf = round(conf * 0.9, 2)

    return label, conf

# ============================================
# TXT PROCESSING
# ============================================

def process_txt(file_path):
    results = []

    with open(file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        match = re.match(r"(Q?\d+)\.\s*(.*)", line)

        if match:
            qno = match.group(1)
            question = match.group(2)

            label, confidence = hybrid_predict(question)
            results.append([qno, question, label, confidence])

    return results

# ============================================
# DISPLAY
# ============================================

def display_results(results):
    df_out = pd.DataFrame(results, columns=["Qno", "Question", "BTL Level", "Confidence"])

    print("\n===== FINAL HYBRID BTL RESULTS =====\n")
    print(df_out.to_string(index=False))

# ============================================
# RUN
# ============================================

file_path = '/kaggle/input/datasets/lakshmir03/btl-que/btl_questions.txt'

results = process_txt(file_path)
display_results(results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Step,Training Loss
10,2.829080
20,1.902574
30,1.795515
40,2.018023
50,2.117479
60,1.932136
70,1.816487
80,1.560068
90,1.360656
100,1.550470


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== FINAL HYBRID BTL RESULTS =====

Qno                                                                             Question  BTL Level  Confidence
 Q1 Which of the following groups or bodies did not offer direct relief to needy people? Understand        0.86
 Q2                          What criteria would you use to assess today's stock market?   Evaluate        0.99
 Q3                                          Can you see a possible solution to famines?     Create        0.99
 Q4                                                  What would result in deforestation? Understand        0.87
 Q5                                                            How many eggs in a dozen?   Remember        0.95
